# Lesson 1: Agent Foundations — The Core Loop

## What You'll Learn

1. **What is an agent?** — The mental model that separates agents from chatbots
2. **PydanticAI basics** — Creating agents, calling LLMs, structured output
3. **Provider switching** — Same code, different LLMs (OpenAI, Claude, Ollama)
4. **Dependency injection** — Pass runtime context to your agent
5. **Cost tracking** — Know what every LLM call costs from day 1

---

## The Big Picture: What Makes an Agent?

A **chatbot** takes a message and returns a response. That's it.

An **agent** has a *loop*:

```
while not done:
    1. OBSERVE  — Look at the current state (user question, data, tool results)
    2. THINK    — Reason about what to do next (LLM call)
    3. ACT      — Execute an action (call a tool, write code, query data)
    4. Check    — Did I get what I need? If yes, respond. If no, loop.
```

The formula: **Agent = LLM + Tools + Memory + Loop**

In this lesson, we start with just the **LLM** part. No tools, no memory, no loop yet.
We'll add each piece in subsequent lessons.

---

## Setup

First, let's load environment variables and verify our API keys are configured.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()


# Load .env from common launch locations (project root or notebooks/)
env_path = None
for candidate in (Path('.env'), Path('../.env')):
    if candidate.exists():
        env_path = candidate
        break

if env_path is not None:
    load_dotenv(env_path)
    print(f"Loaded .env file from {env_path.resolve()}")
else:
    print("No .env file found — copy .env.example to .env and add provider settings")

# LM Studio OpenAI-compatible local setup
if os.getenv('LMSTUDIO_BASE_URL') and not os.getenv('OPENAI_BASE_URL'):
    os.environ['OPENAI_BASE_URL'] = os.getenv('LMSTUDIO_BASE_URL').rstrip('/') + '/v1'
if os.getenv('LMSTUDIO_BASE_URL') and not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = 'lm-studio'

# Check which providers are available
providers = {
    'LM Studio': bool(os.getenv('LMSTUDIO_BASE_URL')),
    'OpenAI': bool(os.getenv('OPENAI_API_KEY')),
    'Anthropic': bool(os.getenv('ANTHROPIC_API_KEY')),
}
print('\nAvailable providers:')
for provider, available in providers.items():
    status = '✅' if available else '❌'
    print(f"  {status} {provider}")


---

## Part 1: Your First Agent — Hello PydanticAI

### Why PydanticAI?

| Feature | PydanticAI | Raw API calls | LangChain |
|---------|------------|---------------|----------|
| Type safety | Built-in (Pydantic models) | Manual | Partial |
| Structured output | First-class | JSON mode + parsing | Via output parsers |
| Tool calling | Decorators with validation | Manual schema building | Lots of abstraction |
| Provider switching | One-line change | Rewrite code | Adapter pattern |
| Learning value | See everything clearly | Too low-level | Too much magic |

PydanticAI sits in the sweet spot: **enough abstraction to be productive, little enough to understand what's happening.**

In [ ]:
from pydantic_ai import Agent

# -----------------------------------------------------------------
# The simplest possible agent — just an LLM with a system prompt.
# No tools, no memory, no loop. We'll add those in later lessons.
# -----------------------------------------------------------------

simple_agent = Agent(
    "openai:local-model",  # <-- change this ONE string to switch providers
    system_prompt="You are a helpful data analyst assistant. Be concise.",
)

# run_sync() is the synchronous version — great for notebooks
result = simple_agent.run_sync("What are the top 3 things to check when exploring a new dataset?")

print(result.output)

### What just happened?

1. `Agent("openai:local-model", ...)` — Created an agent bound to a specific model
2. `system_prompt=...` — Told the LLM its role (this goes in every request)
3. `run_sync("...")` — Sent the user message, got back a result
4. `result.output` — The LLM's text response

The `result` object contains much more than just the output. Let's inspect it.

In [ ]:
# ---- Inspecting the result object ----

# Token usage — critical for cost tracking
usage = result.usage()
print(f"Input tokens:  {usage.input_tokens}")
print(f"Output tokens: {usage.output_tokens}")
print(f"Total tokens:  {usage.input_tokens + usage.output_tokens}")
print(f"Requests made: {usage.requests}")

# Full message history — see exactly what was sent/received
print(f"\nMessage history ({len(result.all_messages())} messages):")
for msg in result.all_messages():
    print(f"  [{msg.kind}] {str(msg)[:100]}...")

### Production Pattern #1: Cost Tracking from Day 1

Every LLM call costs money. In production, you MUST track this.
Let's build a simple cost calculator.

In [ ]:
# -----------------------------------------------------------------
# Cost tracking utility — we'll use this throughout the project.
# Prices as of early 2025. Update as needed.
# -----------------------------------------------------------------

# Pricing per 1M tokens (input, output)
MODEL_PRICING = {
    "gpt-4o-mini": (0.15, 0.60),
    "gpt-4o": (2.50, 10.00),
    "claude-3-5-sonnet-latest": (3.00, 15.00),
    "claude-3-5-haiku-latest": (0.80, 4.00),
    "llama3.2": (0.0, 0.0),  # Local Ollama — free!
}


def estimate_cost(usage, model_name: str) -> dict:
    """Estimate the USD cost of an LLM call."""
    # Extract base model name (strip provider prefix like 'openai:')
    base_name = model_name.split(":")[-1] if ":" in model_name else model_name

    input_price, output_price = MODEL_PRICING.get(base_name, (0.0, 0.0))

    input_cost = (usage.input_tokens / 1_000_000) * input_price
    output_cost = (usage.output_tokens / 1_000_000) * output_price

    return {
        "model": base_name,
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "input_cost_usd": round(input_cost, 6),
        "output_cost_usd": round(output_cost, 6),
        "total_cost_usd": round(input_cost + output_cost, 6),
    }


cost = estimate_cost(result.usage(), "openai:local-model")
print(f"Model: {cost['model']}")
print(f"Tokens: {cost['input_tokens']} in / {cost['output_tokens']} out")
print(f"Cost: ${cost['total_cost_usd']:.6f}")

---

## Part 2: Structured Output — The Agent Superpower

### Why this matters

Raw text output is useless in production. You can't reliably parse "The answer is 42" from free text.

**Structured output** means the LLM returns a **validated Pydantic model** — not a string.
If the LLM returns invalid data, PydanticAI automatically asks it to retry.

This is the single most important pattern in production agentic AI.

In [ ]:
from pydantic import BaseModel, Field


# -----------------------------------------------------------------
# Define the EXACT structure we want the LLM to return.
# The LLM MUST conform to this schema. No wiggle room.
# -----------------------------------------------------------------

class DatasetAssessment(BaseModel):
    """Assessment of a dataset's suitability for analysis."""

    dataset_name: str = Field(description="Name or description of the dataset")
    row_count_estimate: int = Field(description="Estimated number of rows")
    key_columns: list[str] = Field(description="Most important columns for analysis")
    data_quality_issues: list[str] = Field(description="Potential quality problems")
    suggested_analyses: list[str] = Field(description="Top 3 analyses to run")
    complexity_score: int = Field(
        ge=1, le=10,
        description="How complex is this dataset to analyze? 1=trivial, 10=very complex"
    )


# Create an agent that MUST return a DatasetAssessment
assessment_agent = Agent(
    "openai:local-model",
    output_type=DatasetAssessment,  # <-- This is the magic
    retries=5,
    model_settings={"temperature": 0.0},
    system_prompt=(
        "You are a data analyst. When given a dataset description, return ONLY a valid "
        "JSON object that exactly matches the DatasetAssessment schema. "
        "Do not include markdown, prose, or code fences."
    ),
)

result = assessment_agent.run_sync(
    "I have a CSV with columns: date, product, category, region, quantity, "
    "unit_price, revenue, cost. It has about 40 rows of weekly sales data "
    "across 4 regions for different products."
)

# result.output is now a DatasetAssessment object — not a string!
assessment = result.output

print(f"Dataset: {assessment.dataset_name}")
print(f"Complexity: {assessment.complexity_score}/10")
print(f"Key columns: {assessment.key_columns}")
print(f"\nSuggested analyses:")
for i, analysis in enumerate(assessment.suggested_analyses, 1):
    print(f"  {i}. {analysis}")
print(f"\nQuality issues:")
for issue in assessment.data_quality_issues:
    print(f"  - {issue}")

In [ ]:
# -----------------------------------------------------------------
# The output is a real Pydantic model — you get full type safety.
# You can serialize it, validate it, use it in downstream code.
# -----------------------------------------------------------------

# Serialize to dict
print("As dict:")
print(assessment.model_dump())

# Serialize to JSON
print("\nAs JSON:")
print(assessment.model_dump_json(indent=2))

# Type-safe access — your IDE knows these fields exist
if assessment.complexity_score > 7:
    print("\nThis is a complex dataset — consider breaking the analysis into steps.")
else:
    print(f"\nComplexity {assessment.complexity_score}/10 — straightforward to analyze.")

### What makes this different from "just parsing JSON"?

1. **Schema is sent to the LLM** — The LLM sees the Pydantic model's JSON schema and follows it
2. **Validation is automatic** — If the LLM returns `complexity_score: 15` (max is 10), PydanticAI catches it and asks the LLM to fix it
3. **Retries are built-in** — The LLM gets up to 3 chances to produce valid output
4. **No parsing code** — You never write `json.loads()` or regex extraction

---

## Part 3: Provider Switching — One Line to Rule Them All

### The provider-agnostic pattern

In production, you want to:
- **A/B test** models (is Claude better than GPT for your use case?)
- **Fallback** to another provider if one is down
- **Cost optimize** (use cheap models for simple tasks, expensive for complex ones)

PydanticAI makes this trivial — just change the model string.

In [ ]:
from pydantic_ai import Agent
from pydantic import BaseModel, Field


class QuickAnalysis(BaseModel):
    """A quick analysis of a data question."""
    approach: str = Field(description="How you would approach this analysis")
    key_metric: str = Field(description="The single most important metric to compute")
    confidence: float = Field(ge=0.0, le=1.0, description="How confident are you in this approach")


# -----------------------------------------------------------------
# Same question, different providers.
# The model string format is: "provider:model_name"
# -----------------------------------------------------------------

QUESTION = (
    "Given weekly sales data with revenue and cost columns across 4 regions, "
    "what's the best way to find which region is most profitable?"
)

# Define models to test — comment out any you don't have keys for
models_to_test = []

if os.getenv("OPENAI_API_KEY"):
    models_to_test.append("openai:local-model")

if os.getenv("ANTHROPIC_API_KEY"):
    models_to_test.append("anthropic:claude-3-5-haiku-latest")

# Ollama (local) — uncomment if you have Ollama running
# models_to_test.append("ollama:llama3.2")

print(f"Testing {len(models_to_test)} provider(s)...\n")

for model_name in models_to_test:
    agent = Agent(
        model_name,
        output_type=QuickAnalysis,
        system_prompt="You are a senior data analyst. Be precise and concise.",
    )

    result = agent.run_sync(QUESTION)
    analysis = result.output
    usage = result.usage()
    cost = estimate_cost(usage, model_name)

    print(f"{'=' * 60}")
    print(f"Model: {model_name}")
    print(f"Approach: {analysis.approach}")
    print(f"Key metric: {analysis.key_metric}")
    print(f"Confidence: {analysis.confidence:.0%}")
    print(f"Cost: ${cost['total_cost_usd']:.6f} ({cost['input_tokens']}+{cost['output_tokens']} tokens)")
    print()

### Provider-Specific Model Setup

For more control (custom base URLs, API versions, etc.), you can create model objects directly.

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

# -----------------------------------------------------------------
# Example: Connecting to Ollama (local model)
# First run: ollama pull llama3.2
# Then:      ollama serve
# -----------------------------------------------------------------

# Uncomment to use Ollama:
# ollama_model = OpenAIChatModel(
#     model_name="llama3.2",
#     provider=OllamaProvider(base_url="http://localhost:11434/v1"),
# )
# local_agent = Agent(ollama_model, output_type=QuickAnalysis)
# result = local_agent.run_sync(QUESTION)
# print(result.output)

print("Ollama setup shown above — uncomment to use a local model (free, private, no API key).")
print("Install: brew install ollama && ollama pull llama3.2 && ollama serve")

### Fallback Models — Automatic Provider Failover

In production, you don't want your agent to die because OpenAI is having an outage.
PydanticAI has built-in `FallbackModel` support.

In [ ]:
from pydantic_ai.models.fallback import FallbackModel
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.models.anthropic import AnthropicModel

# -----------------------------------------------------------------
# FallbackModel: tries providers in order. If one fails, moves to next.
# This is how production agents stay reliable.
# -----------------------------------------------------------------

# Only build fallback if we have multiple providers
available_models = []
if os.getenv("OPENAI_API_KEY"):
    available_models.append(OpenAIChatModel("gpt-4o-mini"))
if os.getenv("ANTHROPIC_API_KEY"):
    available_models.append(AnthropicModel("claude-3-5-haiku-latest"))

if len(available_models) >= 2:
    fallback = FallbackModel(*available_models)
    resilient_agent = Agent(
        fallback,
        output_type=QuickAnalysis,
        system_prompt="You are a senior data analyst.",
    )
    result = resilient_agent.run_sync(QUESTION)
    print(f"Response from fallback chain:")
    print(f"  Approach: {result.output.approach}")
    # Check which model actually responded
    for msg in result.all_messages():
        if hasattr(msg, 'model_name') and msg.model_name:
            print(f"  Served by: {msg.model_name}")
            break
else:
    print("Need 2+ providers to demo fallback. Add more API keys to .env")

---

## Part 4: System Prompts & Dependency Injection

### Dynamic System Prompts

Static system prompts are fine for simple cases. But a data analyst agent needs to know
about the *specific* dataset it's working with. That's dynamic.

PydanticAI uses **dependency injection** — you pass runtime data to the agent,
and it becomes available in system prompts and tools via `RunContext`.

In [ ]:
import pandas as pd
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext


# -----------------------------------------------------------------
# Dependencies: runtime data that the agent needs access to.
# This is NOT part of the prompt — it's injected at runtime.
# -----------------------------------------------------------------

@dataclass
class AnalystDeps:
    """Runtime dependencies for the data analyst agent."""
    dataset_name: str
    df: pd.DataFrame
    max_cost_usd: float = 0.05  # Budget per query


analyst = Agent(
    "openai:local-model",
    deps_type=AnalystDeps,
    output_type=DatasetAssessment,
    retries=5,
    model_settings={"temperature": 0.0},
    system_prompt=(
        "You are a data analyst. Return ONLY a valid JSON object matching "
        "the DatasetAssessment schema exactly."
    ),
)


# Dynamic system prompt — runs at each agent call, has access to dependencies
@analyst.system_prompt
def add_dataset_context(ctx: RunContext[AnalystDeps]) -> str:
    """Inject dataset metadata into the system prompt."""
    df = ctx.deps.df
    return (
        f"\nDataset: {ctx.deps.dataset_name}\n"
        f"Shape: {df.shape[0]} rows x {df.shape[1]} columns\n"
        f"Columns: {', '.join(df.columns.tolist())}\n"
        f"Dtypes: {df.dtypes.to_dict()}\n"
        f"Sample (first 3 rows):\n{df.head(3).to_string()}\n"
        f"\nBasic stats:\n{df.describe().to_string()}"
    )


# Load our sample data and run
sales_df = pd.read_csv("../data/sample_sales.csv")
deps = AnalystDeps(dataset_name="Weekly Sales Data", df=sales_df)

result = analyst.run_sync("Assess this dataset for analysis.", deps=deps)

assessment = result.output
print(f"Dataset: {assessment.dataset_name}")
print(f"Rows: ~{assessment.row_count_estimate}")
print(f"Complexity: {assessment.complexity_score}/10")
print(f"\nKey columns: {assessment.key_columns}")
print(f"\nSuggested analyses:")
for a in assessment.suggested_analyses:
    print(f"  - {a}")

cost = estimate_cost(result.usage(), "openai:local-model")
print(f"\nCost: ${cost['total_cost_usd']:.6f}")

### Why Dependency Injection Matters

Without DI, you'd stuff everything into the user prompt as a giant string. That's fragile.

With DI:
- **System prompts** can access the DataFrame to describe it
- **Tools** can access the DataFrame to query it (Lesson 2)
- **Validators** can access cost limits to enforce budgets
- **Everything is typed** — your IDE knows `ctx.deps` is an `AnalystDeps`

---

## Part 5: Building a Model Router — Smart Provider Selection

In production, you want to route queries to different models based on complexity:
- Simple questions → cheap model (gpt-4o-mini / haiku)
- Complex analysis → powerful model (gpt-4o / sonnet)

This is a common production pattern that saves significant costs.

In [ ]:
from pydantic_ai import Agent
from pydantic import BaseModel, Field


class QueryClassification(BaseModel):
    """Classification of a user query's complexity."""
    complexity: str = Field(description="'simple' or 'complex'")
    reason: str = Field(description="Why this complexity level")


# Use the cheapest model to classify query complexity
classifier = Agent(
    "openai:local-model",
    output_type=QueryClassification,
    system_prompt=(
        "Classify the user's data analysis question as 'simple' or 'complex'.\n"
        "Simple: single aggregation, filtering, basic stats, direct lookups.\n"
        "Complex: multi-step analysis, correlations, time series, comparisons across groups."
    ),
)


def route_to_model(query: str) -> str:
    """Route a query to the appropriate model based on complexity."""
    result = classifier.run_sync(query)
    classification = result.output

    if classification.complexity == "simple":
        model = "openai:local-model"  # Cheap and fast
    else:
        model = "openai:gpt-4o"  # Powerful but expensive

    return model, classification


# Test with different queries
queries = [
    "What is the total revenue?",
    "Which region has the highest profit margin, and how has it trended over time compared to others?",
    "How many rows are in the dataset?",
    "Build a forecast model for next quarter's revenue by region, accounting for seasonal patterns.",
]

for q in queries:
    model, classification = route_to_model(q)
    print(f"[{classification.complexity:>7}] {model:<25} | {q[:70]}")

---

## Part 6: Async Patterns — Production-Ready Execution

In production, your agent runs inside a web server (FastAPI). Blocking calls
(`run_sync`) would kill your throughput. PydanticAI is async-first.

In [ ]:
import asyncio
from pydantic_ai import Agent
from pydantic import BaseModel, Field


class ColumnAnalysis(BaseModel):
    column_name: str
    data_type: str
    description: str
    usefulness_score: int = Field(ge=1, le=10)


column_agent = Agent(
    "openai:local-model",
    output_type=ColumnAnalysis,
    system_prompt="Analyze a single column of a sales dataset. Rate its usefulness for business insights.",
)


async def analyze_columns_concurrently(columns: list[str]) -> list[ColumnAnalysis]:
    """Analyze multiple columns in parallel — much faster than sequential."""
    tasks = [
        column_agent.run(f"Analyze this column from a sales dataset: '{col}'")
        for col in columns
    ]
    results = await asyncio.gather(*tasks)
    return [r.output for r in results]


# Run concurrent analysis
columns = ["date", "product", "region", "revenue", "cost"]
analyses = await analyze_columns_concurrently(columns)

print(f"Analyzed {len(analyses)} columns concurrently:\n")
for a in sorted(analyses, key=lambda x: x.usefulness_score, reverse=True):
    print(f"  [{a.usefulness_score:>2}/10] {a.column_name:<15} ({a.data_type}) — {a.description[:60]}")

### Why async matters

- **Concurrency**: Analyze 5 columns in parallel = ~1 API call time instead of 5
- **Server-ready**: `agent.run()` (async) works directly in FastAPI handlers
- **Streaming**: Async enables token-by-token streaming (Lesson 8)

In notebooks, we can use `await` directly. In scripts, wrap in `asyncio.run()`.

---

## Summary & Key Takeaways

### What we built
| Component | What | Why |
|-----------|------|-----|
| Basic agent | `Agent(model, system_prompt)` | The atomic unit of agentic AI |
| Structured output | `output_type=PydanticModel` | Production-safe, typed, validated |
| Provider switching | Change model string | A/B testing, cost optimization, resilience |
| Fallback model | `FallbackModel(a, b)` | Automatic failover between providers |
| Dependency injection | `deps_type`, `RunContext` | Pass runtime data to prompts & tools |
| Model router | Classify → route | Smart cost optimization |
| Async execution | `agent.run()` + `asyncio.gather` | Concurrent, server-ready |
| Cost tracking | Token counting | Know what you're spending |

### What's missing (coming in next lessons)
- **Tools** (Lesson 2) — Let the agent call functions and query data
- **Code execution** (Lesson 3) — Let the agent write and run Python code
- **Reasoning loop** (Lesson 4) — Multi-step problem solving
- **Memory** (Lesson 5) — Remember context across turns

### Production patterns introduced
1. `.env`-based configuration (never hardcode API keys)
2. Cost tracking from day 1
3. Structured output for reliability
4. Provider fallback for resilience
5. Model routing for cost optimization
6. Async-first for scalability

---

## Exercises

1. **Add a new Pydantic model** — Create a `DataQualityReport` with fields like `missing_value_columns`, `duplicate_row_count`, `outlier_columns`. Run the agent with the employees dataset.

2. **Compare providers** — If you have both OpenAI and Anthropic keys, run the same structured output query on both and compare: accuracy, cost, latency.

3. **Try Ollama** — Install Ollama (`brew install ollama`), pull a model (`ollama pull llama3.2`), and test the same queries locally. Note the quality vs cost tradeoff.

4. **Build a cost tracker** — Create a class that accumulates costs across multiple agent calls and prints a summary at the end.